In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                    TOY STABLE DIFFUSION TRAINING PIPELINE                    ║
║                        Cell 1: Environment Setup & Imports                   ║
╚═══════════════════════════════════════════════════════════════════════════════╝

A production-ready training pipeline for building a Stable Diffusion model from
scratch on a single T4 GPU (16GB VRAM).

Author: Your Learning Journey
Target: 512x512 image generation with optimized architecture
Dataset: LAION (Refined) subset (168K images with captions)
"""

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1: MOUNT GOOGLE DRIVE & SETUP PATHS
# ═══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
import os
import zipfile
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define base paths
PROJECT_ROOT = "/content/drive/MyDrive/ToyStableDiffusion"
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
LOGS_DIR = os.path.join(PROJECT_ROOT, "logs")
SAMPLES_DIR = os.path.join(PROJECT_ROOT, "samples")
# ZIP_PATH = os.path.join(PROJECT_ROOT, "laion_512.zip")
LOCAL_DATA_DIR = "/content/laion_512"  # Local to Colab for faster I/O

# Create directories
for dir_path in [PROJECT_ROOT, CHECKPOINTS_DIR, LOGS_DIR, SAMPLES_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print(f"✓ Google Drive mounted successfully")
print(f"✓ Project root: {PROJECT_ROOT}")

# Check if data is already ready
if not os.path.exists(LOCAL_DATA_DIR) or not os.path.exists(os.path.join(LOCAL_DATA_DIR, "images")):
    print(f"\n📦 Dataset not found. Starting High-Speed Hugging Face Download...")

    # 1. Enable HF Transfer for speed
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    # 2. Download the Tar file to /content
    REPO_ID = "dheeren-tejani/laion_512"
    FILENAME = "laion_512.tar"
    TAR_PATH = f"/content/{FILENAME}"

    print(f"   Downloading {FILENAME} from {REPO_ID}...")
    !hf download {REPO_ID} {FILENAME} --local-dir /content --repo-type dataset

    # 3. Untar (Extract)
    print(f"   Extracting {FILENAME}...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    !tar -xf {TAR_PATH} -C {LOCAL_DATA_DIR}

    # 4. Clean up Tar file to save space
    if os.path.exists(TAR_PATH):
        os.remove(TAR_PATH)
        print("   ✓ Cleaned up compressed archive.")

    # ═══════════════════════════════════════════════════════════════════════════
    # SECTION 3: AUTOMATIC FOLDER FIX (Replaces the "Small Cell")
    # ═══════════════════════════════════════════════════════════════════════════
    # Sometimes tar extracts into a subfolder (e.g., laion_512/LAION_AESTHETIC_512)
    # This block detects that and moves everything up one level automatically.

    nested_folders = [d for d in os.listdir(LOCAL_DATA_DIR) if os.path.isdir(os.path.join(LOCAL_DATA_DIR, d))]

    # If we see a nested folder like "LAION_AESTHETIC_512" but no "images" folder in root
    if len(nested_folders) == 1 and not os.path.exists(os.path.join(LOCAL_DATA_DIR, "images")):
        nested_path = os.path.join(LOCAL_DATA_DIR, nested_folders[0])
        print(f"🕵️ Detect nested structure: {nested_path}. Fixing...")

        # Move all contents up
        for item in os.listdir(nested_path):
            shutil.move(os.path.join(nested_path, item), LOCAL_DATA_DIR)

        # Remove empty nested folder
        os.rmdir(nested_path)
        print("   ✓ File structure fixed.")

    print("\n✅ Dataset Ready!")
else:
    print("\n✅ Dataset already exists at local path. Skipping download.")

# Verify structure for Cell 2
print(f"   Images found: {os.path.exists(os.path.join(LOCAL_DATA_DIR, 'images'))}")
print(f"   JSON found: {os.path.exists(os.path.join(LOCAL_DATA_DIR, 'annotations.json'))}")
print(f"   Tokenizer found: {os.path.exists(os.path.join(LOCAL_DATA_DIR, 'tokenizer.pt'))}")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2: INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

#!pip install torch torchvision torchaudio
#!pip install transformers datasets
!pip install bitsandbytes  # For 8-bit optimizers
#!pip install accelerate
#!pip install wandb  # For experiment tracking
#!pip install einops  # For tensor operations
#!pip install pillow numpy matplotlib tqdm
!pip install ftfy # regex For text processing
!pip install kornia  # For GPU-accelerated augmentations

print("✓ All dependencies installed")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3: IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
from torchvision.utils import save_image, make_grid

import bitsandbytes as bnb
from transformers import get_cosine_schedule_with_warmup
from datasets import load_dataset

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import json
import time
from datetime import datetime
from typing import Optional, Tuple, List, Dict, Any
from dataclasses import dataclass, field
from pathlib import Path
from tqdm.auto import tqdm
import math
import random
import gc
import warnings
import glob
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4: UNIFIED CONFIGURATION CLASS
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class UnifiedConfig:
    """Master configuration for all components and training"""

    # ═══════════════════════════════════════════════════════════════════════════
    # VAE CONFIGURATION (512px → 64px latent)
    # ═══════════════════════════════════════════════════════════════════════════
    vae_image_size: int = 512
    vae_in_channels: int = 3
    vae_latent_dim: int = 4  # Standard for Stable Diffusion
    vae_latent_size: int = 64  # 512 / 8 = 64
    vae_hidden_dims: List[int] = field(default_factory=lambda: [128, 256, 512])
    vae_beta: float = 0.000001  # KL divergence weight

    # ═══════════════════════════════════════════════════════════════════════════
    # CLIP CONFIGURATION (Text-Image Encoder)
    # ═══════════════════════════════════════════════════════════════════════════
    clip_vocab_size: int = 49408  # LAION tokenizer size
    clip_embed_dim: int = 512
    clip_num_layers: int = 12
    clip_num_heads: int = 8
    clip_mlp_ratio: int = 4
    clip_max_seq_length: int = 77
    clip_dropout: float = 0.1

    # Image encoder config
    clip_image_size: int = 256
    clip_patch_size: int = 16
    clip_vision_layers: int = 12

    # ═══════════════════════════════════════════════════════════════════════════
    # UNET CONFIGURATION (Diffusion Model)
    # ═══════════════════════════════════════════════════════════════════════════
    unet_image_size: int = 64  # Latent space size
    unet_in_channels: int = 4  # Latent channels
    unet_out_channels: int = 4
    unet_model_channels: int = 192
    unet_num_res_blocks: int = 2
    unet_attention_resolutions: Tuple[int] = (16, 8)
    unet_channel_mult: Tuple[int] = (1, 2, 2, 4)
    unet_dropout: float = 0.1
    unet_num_heads: int = 3
    unet_context_dim: int = 512  # CLIP embedding dim
    unet_use_checkpoint: bool = True  # Gradient checkpointing for VRAM

    # ═══════════════════════════════════════════════════════════════════════════
    # TRAINING CONFIGURATION
    # ═══════════════════════════════════════════════════════════════════════════
    component: str = "unet"  # "vae", "clip", or "unet"
    batch_size: int = 32
    num_epochs: int = 50
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_steps: int = 2000

    # Optimization
    use_8bit_adam: bool = True
    gradient_clip: float = 0.5  # Safety rail
    use_amp: bool = True  # Automatic Mixed Precision
    accumulation_steps: int = 4  # Effective batch = 128

    # Checkpointing & Logging
    save_every: int = 1000
    sample_every: int = 1000
    log_every: int = 200
    num_samples: int = 4
    max_checkpoints: int = 2  # Sliding window for checkpoints

    # EMA (Exponential Moving Average)
    use_ema: bool = True
    ema_decay: float = 0.9999

    # Diffusion specific
    num_diffusion_steps: int = 1000
    cfg_dropout: float = 0.1  # Classifier-free guidance dropout

    # Paths
    checkpoint_path: Optional[str] = None
    vae_path: Optional[str] = None
    clip_path: Optional[str] = None
    unet_path: Optional[str] = None

    # Flow Matching configuration
    use_flow_matching: bool = True  # 🔥 NEW: Toggle Flow Matching
    flow_sigma_min: float = 0.0     # Minimum noise (0 = pure flow matching)
    flow_inference_steps: int = 20

# Create global config
config = UnifiedConfig()

print("✓ Unified Configuration initialized")
print(f"  Component: {config.component}")
print(f"  VAE: {config.vae_image_size}px → {config.vae_latent_size}px latent")
print(f"  CLIP: {config.clip_vocab_size} vocab, {config.clip_embed_dim}D embeddings")
print(f"  U-Net: {config.unet_model_channels} base channels")
print(f"  Batch Size: {config.batch_size} (effective: {config.batch_size * config.accumulation_steps})")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5: UTILITY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

class Logger:
    """Comprehensive logging utility"""
    def __init__(self, log_dir: str, experiment_name: str):
        self.log_dir = Path(log_dir)
        self.experiment_name = experiment_name
        self.log_file = self.log_dir / f"{experiment_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        self.metrics = []

    def log(self, message: str, level: str = "INFO"):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_message = f"[{timestamp}] [{level}] {message}"
        print(log_message)
        with open(self.log_file, "a") as f:
            f.write(log_message + "\n")

    def log_metrics(self, step: int, metrics: Dict[str, float]):
        metrics["step"] = step
        metrics["timestamp"] = time.time()
        self.metrics.append(metrics)

        # Log to console
        metrics_str = " | ".join([f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}"
                                  for k, v in metrics.items()])
        self.log(f"Step {step} - {metrics_str}")

        # Save metrics to JSON
        with open(self.log_dir / f"{self.experiment_name}_metrics.json", "w") as f:
            json.dump(self.metrics, f, indent=2)

    def plot_metrics(self, metric_names: List[str]):
        """Plot training curves"""
        fig, axes = plt.subplots(1, len(metric_names), figsize=(6*len(metric_names), 4))
        if len(metric_names) == 1:
            axes = [axes]

        for ax, metric_name in zip(axes, metric_names):
            steps = [m["step"] for m in self.metrics if metric_name in m]
            values = [m[metric_name] for m in self.metrics if metric_name in m]
            ax.plot(steps, values)
            ax.set_xlabel("Step")
            ax.set_ylabel(metric_name)
            ax.set_title(f"{metric_name} over training")
            ax.grid(True)

        plt.tight_layout()
        plt.savefig(self.log_dir / f"{self.experiment_name}_curves.png", dpi=150)
        plt.close()

def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def format_time(seconds: float) -> str:
    """Format seconds into human-readable time"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def manage_checkpoints(component: str, current_step: int, max_keep: int = 3):
    """
    Sliding window checkpoint management.
    Keeps only the latest N checkpoints to save storage.
    """
    pattern = os.path.join(CHECKPOINTS_DIR, f"{component}_step_*.pt")
    checkpoints = glob.glob(pattern)

    if len(checkpoints) <= max_keep:
        return

    # Sort by step number
    checkpoint_steps = []
    for ckpt in checkpoints:
        try:
            step = int(ckpt.split("_step_")[-1].split(".")[0])
            checkpoint_steps.append((step, ckpt))
        except ValueError:
            continue

    checkpoint_steps.sort(key=lambda x: x[0])

    # Delete old checkpoints (keep only max_keep newest)
    to_delete = checkpoint_steps[:-max_keep]
    for step, path in to_delete:
        try:
            os.remove(path)
            print(f"🗑️  Deleted old checkpoint: {os.path.basename(path)}")
        except Exception as e:
            print(f"⚠️  Failed to delete {path}: {e}")

def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    epoch: int,
    step: int,
    loss: float,
    path: str,
    component: str,
    ema_model: Optional[nn.Module] = None,
    scaler: Optional[GradScaler] = None
):
    """Save comprehensive checkpoint with sliding window management"""
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "epoch": epoch,
        "step": step,
        "loss": loss,
        "scaler_state_dict": scaler.state_dict() if scaler else None,
    }

    if ema_model is not None:
        checkpoint["ema_model_state_dict"] = ema_model.state_dict()

    torch.save(checkpoint, path)
    print(f"✓ Checkpoint saved: {path}")

    # Manage checkpoints (sliding window)
    if "_step_" in path:  # Only manage step checkpoints, not "best" or "final"
        manage_checkpoints(component, step, config.max_checkpoints)

def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    ema_model: Optional[nn.Module] = None,
    scaler: Optional[GradScaler] = None
) -> Tuple[int, int, float]:
    """Safe Load Checkpoint (Robust to missing keys)"""
    checkpoint = torch.load(path, map_location=device)

    # 1. Load Model (Strict - must exist)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        # Fallback for our rescued file which might be the state_dict itself
        model.load_state_dict(checkpoint)

    # 2. Load Optimizer (Lenient - okay if missing)
    if optimizer and checkpoint.get("optimizer_state_dict"):
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        except:
            print("⚠️ Warning: Could not load optimizer state. Starting fresh.")

    # 3. Load Scheduler (Lenient - okay if missing)
    if scheduler and checkpoint.get("scheduler_state_dict"):
        try:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        except:
            print("⚠️ Warning: Could not load scheduler state. Starting fresh.")

    # 4. Load EMA (Lenient)
    if ema_model and checkpoint.get("ema_model_state_dict"):
        ema_model.load_state_dict(checkpoint["ema_model_state_dict"])

    # 5. Load Scaler (Lenient)
    if scaler and checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    step = checkpoint.get("step", 0)
    loss = checkpoint.get("loss", float('inf'))

    print(f"✓ Checkpoint loaded: {path}")
    print(f"  Resuming from epoch {epoch}, step {step}, loss {loss:.6f}")

    return epoch, step, loss

class EMA:
    """Exponential Moving Average for model weights"""
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        # Initialize shadow weights
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def state_dict(self):
        """Return the shadow weights dict"""
        return self.shadow

    def load_state_dict(self, state_dict):
        """Load the shadow weights dict"""
        self.shadow = state_dict

    def update(self):
        """Update EMA weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                assert name in self.shadow
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        """Apply EMA weights to model (for inference)"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        """Restore original weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

print("✓ Utility functions and Logger class defined")
print("\n" + "="*80)
print(config.save_every, config.sample_every)
print("SETUP COMPLETE - Ready to proceed to Cell 2 (Dataset)")
print("="*80)

In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                      Cell 2: Dataset & Tokenizer (LAION)                     ║
╚═══════════════════════════════════════════════════════════════════════════════╝

Loads the LAION dataset and tokenizer from the unzipped local directory.
"""

import json
from PIL import Image
import re
from collections import Counter

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1: LOAD TOKENIZER
# ═══════════════════════════════════════════════════════════════════════════════

TOKENIZER_PATH = os.path.join(LOCAL_DATA_DIR, "tokenizer.pt")

class SimpleTokenizer:
    def __init__(self, vocab_size: int = 49408, max_length: int = 77):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.word2idx = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.vocab_built = False

    def build_vocab(self, captions: list):
        word_freq = Counter()
        for caption in tqdm(captions, desc="Building vocab"):
            words = re.findall(r'\b\w+\b', caption.lower())
            word_freq.update(words)
        most_common = word_freq.most_common(self.vocab_size - len(self.word2idx))
        for word, _ in most_common:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        self.vocab_built = True
        print(f"✓ Vocabulary built: {len(self.word2idx)} words")

    def encode(self, text: str) -> torch.Tensor:
        words = re.findall(r'\b\w+\b', text.lower())
        ids = [self.word2idx["<SOS>"]]
        for word in words[:self.max_length - 2]:
            ids.append(self.word2idx.get(word, self.word2idx["<UNK>"]))
        ids.append(self.word2idx["<EOS>"])
        ids += [self.word2idx["<PAD>"]] * (self.max_length - len(ids))
        return torch.tensor(ids[:self.max_length], dtype=torch.long)

    def decode(self, ids: torch.Tensor) -> str:
        words = []
        for idx in ids:
            idx = idx.item() if torch.is_tensor(idx) else idx
            word = self.idx2word.get(idx, "<UNK>")
            if word in ["<PAD>", "<SOS>", "<EOS>"]: continue
            words.append(word)
        return " ".join(words)

print("="*80)
print("Loading Tokenizer...")
print("="*80)

if os.path.exists(TOKENIZER_PATH):
    print(f"✓ Loading Tokenizer from {TOKENIZER_PATH}...")
    t_data = torch.load(TOKENIZER_PATH, map_location=device)
    tokenizer = SimpleTokenizer(vocab_size=config.clip_vocab_size, max_length=config.clip_max_seq_length)
    tokenizer.word2idx = t_data["word2idx"]
    tokenizer.idx2word = t_data["idx2word"]
    tokenizer.vocab_built = True
    print(f"✓ Tokenizer loaded with {len(tokenizer.word2idx)} words.")
else:
    raise FileNotFoundError("❌ Tokenizer not found inside the zip! Did you include it?")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2: DATASET CLASS
# ═══════════════════════════════════════════════════════════════════════════════

class LaionDataset(Dataset):
    """Dataset for LAION image-text pairs"""
    def __init__(self, root_dir: str, transform=None, mode: str = "train", val_split: float = 0.05):
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode
        self.img_dir = os.path.join(root_dir, "images")

        # Load JSON annotations
        annotations_file = os.path.join(root_dir, "annotations.json")
        if not os.path.exists(annotations_file):
            raise FileNotFoundError(f"❌ annotations.json not found at {annotations_file}")

        with open(annotations_file, 'r') as f:
            all_data = json.load(f)

        # Split into train/val
        num_val = int(len(all_data) * val_split)
        if mode == "val":
            self.data = all_data[:num_val]
        else:
            self.data = all_data[num_val:]

        print(f"✓ {mode.upper()} dataset initialized: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load Image
        img_path = os.path.join(self.img_dir, item['image_filename'])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Fallback for corrupt files (rare)
            return self.__getitem__((idx + 1) % len(self))

        # Get Caption (take first caption if multiple)
        caption = item['captions'][0] if isinstance(item['captions'], list) else item['captions']

        if self.transform:
            image = self.transform(image)

        return {"image": image, "caption": caption, "image_id": item.get('image_id', idx)}

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3: TRANSFORMS & DATA LOADERS
# ═══════════════════════════════════════════════════════════════════════════════

def get_transforms(component: str):
    """
    Minimal CPU transforms - images are already 256x256.

    CPU Job: Just decode JPEG → Tensor. No math, no resizing.
    All heavy operations (normalization, augmentation) moved to GPU via Kornia.

    This eliminates ~25M floating point ops per batch from the CPU.
    """
    return T.Compose([
        T.ToTensor(),  # Only converts pixels to [0,1] tensor
    ])

def get_dataloader(component: str, batch_size: int, num_workers: int = 2):
    transform = get_transforms(component)
    train_dataset = LaionDataset(LOCAL_DATA_DIR, transform=transform, mode="train")
    val_dataset = LaionDataset(LOCAL_DATA_DIR, transform=transform, mode="val")
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, drop_last=False
    )
    return train_loader, val_loader

print("\n" + "="*80)
print("Testing dataloader...")
print("="*80)

# Test dataloader
try:
    test_loader, _ = get_dataloader("vae", batch_size=4)
    test_batch = next(iter(test_loader))
    print(f"✓ Batch loaded successfully!")
    print(f"  Images shape: {test_batch['image'].shape}")
    print(f"  First caption: {test_batch['caption'][0][:80]}...")
except Exception as e:
    print(f"❌ Dataloader Test Failed: {e}")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4: VISUALIZE SAMPLES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("Visualizing sample images...")
print("="*80)

try:
    sample_loader, _ = get_dataloader("vae", batch_size=8)
    sample_batch = next(iter(sample_loader))
    sample_images = (sample_batch['image'] + 1) / 2
    sample_captions = sample_batch['caption']
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for idx in range(8):
        img = sample_images[idx].permute(1, 2, 0).cpu().numpy()
        caption = sample_captions[idx][:50]
        axes[idx].imshow(img)
        axes[idx].set_title(caption, fontsize=8)
        axes[idx].axis('off')
    plt.suptitle("Sample Images from LAION Dataset", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLES_DIR, "dataset_samples.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Sample visualization saved to:", os.path.join(SAMPLES_DIR, "dataset_samples.png"))
except Exception as e:
    print(f"⚠️ Visualization skipped: {e}")

print("\n" + "="*80)
print("DATASET PREPARATION COMPLETE")
print("="*80)
print(f"✓ Local data: {LOCAL_DATA_DIR}")
print(f"✓ Tokenizer vocab size: {len(tokenizer.word2idx)}")
print("\n✅ Ready for training! Proceed to Cell 3 (Model Architectures)")
print("="*80)

In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                      Cell 3: Model Architectures (Updated)                   ║
╚═══════════════════════════════════════════════════════════════════════════════╝

Defines all three model architectures with safety improvements:
1. VAE (256px → 32px latent) - Safe numerical stability
2. CLIP (Text & Image Encoders) - FP32 attention stability
3. U-Net (Diffusion Model) - Gradient checkpointing + stability
"""

from einops import rearrange
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
import kornia as K
import kornia.augmentation as KA

# ═══════════════════════════════════════════════════════════════════════════════
# GPU AUGMENTATION PIPELINE (Kornia)
# ═══════════════════════════════════════════════════════════════════════════════

class GPUAugmentations(nn.Module):
    """
    Kornia-based GPU augmentation pipeline.

    Offloads 25 million math operations per batch from CPU to GPU:
    - Normalization: (pixel - mean) / std for 256×256×3×batch_size
    - Color jitter: Complex HSV transforms (CLIP only)
    - Flips: Fast tensor operations

    Performance gain: ~30-40% faster data pipeline.
    """
    def __init__(self, component: str):
        super().__init__()
        self.component = component

        # 1. VAE: Just Normalize (Fast)
        if component == "vae":
            self.aug = nn.Sequential(
                KA.Normalize(mean=torch.tensor([0.5, 0.5, 0.5]),
                            std=torch.tensor([0.5, 0.5, 0.5]))
            )

        # 2. CLIP: Jitter + Normalize (Heavy - most expensive on CPU)
        elif component == "clip":
            self.aug = nn.Sequential(
                K.geometry.Resize((256, 256)),
                KA.RandomHorizontalFlip(p=0.5),
                KA.ColorJitter(
                    brightness=0.1,
                    contrast=0.1,
                    saturation=0.1,
                    hue=0.1,
                    p=0.8
                ),
                KA.Normalize(mean=torch.tensor([0.485, 0.456, 0.406]),
                            std=torch.tensor([0.229, 0.224, 0.225]))
            )

        # 3. UNet: Flip + Normalize (Medium)
        elif component == "unet":
            self.aug = nn.Sequential(
                KA.RandomHorizontalFlip(p=0.5),
                KA.Normalize(mean=torch.tensor([0.5, 0.5, 0.5]),
                            std=torch.tensor([0.5, 0.5, 0.5]))
            )
        else:
            raise ValueError(f"Unknown component: {component}")

    def forward(self, x):
        """
        Args:
            x: Tensor of shape (B, C, H, W) in range [0, 1]
        Returns:
            Augmented and normalized tensor
        """
        return self.aug(x)

# ═══════════════════════════════════════════════════════════════════════════════
# PART 1: VAE (VARIATIONAL AUTOENCODER) - SAFE VERSION
# ═══════════════════════════════════════════════════════════════════════════════

class ResidualBlock(nn.Module):
    """Residual block with BatchNorm"""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm1 = nn.BatchNorm2d(in_channels)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.act = nn.LeakyReLU(0.2)
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

class VariationalEncoder(nn.Module):
    """VAE Encoder: 256x256 → 32x32 latent"""
    def __init__(self, in_channels: int = 3, hidden_dims: list = [64, 128, 256], latent_dim: int = 4):
        super().__init__()

        modules = []
        curr_channels = in_channels

        # Build encoder layers (64 → 128 → 256)
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(curr_channels, h_dim, kernel_size=3, stride=2, padding=1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU(0.2)
                )
            )
            curr_channels = h_dim

        self.encoder = nn.Sequential(*modules)

        # Latent projections (1x1 convs to maintain spatial dims)
        self.fc_mu = nn.Conv2d(hidden_dims[-1], latent_dim, 1)
        self.fc_logvar = nn.Conv2d(hidden_dims[-1], latent_dim, 1)

    def forward(self, x):
        x = self.encoder(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        # 🛡️ SAFETY RAIL: Clamp LogVar to prevent exp() overflow
        logvar = torch.clamp(logvar, min=-6.0, max=2.0)

        return mu, logvar

class Decoder(nn.Module):
    """VAE Decoder: 32x32 latent → 256x256"""
    def __init__(self, latent_dim: int = 4, hidden_dims: list = [64, 128, 256], out_channels: int = 3):
        super().__init__()

        # Reverse hidden dims (256 → 128 → 64)
        hidden_dims = list(reversed(hidden_dims))

        # Initial projection
        self.decoder_input = nn.Conv2d(latent_dim, hidden_dims[0], 3, padding=1)

        modules = []
        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i], hidden_dims[i+1],
                                      kernel_size=3, stride=2, padding=1, output_padding=1),
                    nn.BatchNorm2d(hidden_dims[i+1]),
                    nn.LeakyReLU(0.2)
                )
            )

        self.decoder = nn.Sequential(*modules)

        # Final layer (64 → 3 RGB)
        self.final_layer = nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[-1], hidden_dims[-1],
                              kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(hidden_dims[-1]),
            nn.LeakyReLU(0.2),
            nn.Conv2d(hidden_dims[-1], out_channels, kernel_size=3, padding=1),
            nn.Tanh()  # Images are -1 to 1
        )

    def forward(self, z):
        x = self.decoder_input(z)
        x = self.decoder(x)
        x = self.final_layer(x)
        return x

class VAE(nn.Module):
    """Complete VAE: 256x256 ↔ 32x32 latent space"""
    def __init__(self):
        super().__init__()
        self.encoder = VariationalEncoder(
            in_channels=config.vae_in_channels,
            hidden_dims=config.vae_hidden_dims,
            latent_dim=config.vae_latent_dim
        )
        self.decoder = Decoder(
            latent_dim=config.vae_latent_dim,
            hidden_dims=config.vae_hidden_dims,
            out_channels=config.vae_in_channels
        )
        self.free_bits = 0.5  # Prevent posterior collapse

    def reparameterize(self, mu, logvar):
        """Reparameterization trick with numerical stability"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        """Encode to latent space"""
        mu, logvar = self.encoder(x)

        # 🛡️ SAFETY: Clamp mu to prevent extreme values
        mu = torch.clamp(mu, -10, 10)

        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def decode(self, z):
        """Decode from latent space"""
        return self.decoder(z)

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        recon = self.decode(z)
        return recon, mu, logvar

# ═══════════════════════════════════════════════════════════════════════════════
# PART 2: CLIP (TEXT & IMAGE ENCODERS) - STABLE VERSION
# ═══════════════════════════════════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    """Multi-head attention with FP32 stability"""
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, N, C = x.shape

        # Compute Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # 🛡️ SAFETY RAIL: Force Attention to FP32
        with torch.cuda.amp.autocast(enabled=False):
            q, k = q.float(), k.float()

            # Attention
            attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)

            if mask is not None:
                attn = attn.masked_fill(mask == 0, float('-inf'))

            attn = F.softmax(attn, dim=-1)

            # Cast back to original dtype
            attn = attn.to(v.dtype)

        attn = self.dropout(attn)

        # Combine heads
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)

        return x

class TransformerBlock(nn.Module):
    """Transformer block"""
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

        mlp_dim = embed_dim * mlp_ratio
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.mlp(self.norm2(x))
        return x

class TextEncoder(nn.Module):
    """CLIP Text Encoder"""
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(config.clip_vocab_size, config.clip_embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, config.clip_max_seq_length, config.clip_embed_dim))

        self.transformer = nn.ModuleList([
            TransformerBlock(config.clip_embed_dim, config.clip_num_heads,
                           config.clip_mlp_ratio, config.clip_dropout)
            for _ in range(config.clip_num_layers)
        ])

        self.norm = nn.LayerNorm(config.clip_embed_dim)

    def forward(self, text_ids):
        B, seq_len = text_ids.shape

        # Embeddings
        x = self.token_embedding(text_ids)
        x = x + self.pos_embedding[:, :seq_len, :]

        # Transformer
        for block in self.transformer:
            x = block(x)

        x = self.norm(x)

        # Take the [SOS] token embedding (first token)
        text_features = x[:, 0, :]

        # Normalize
        text_features = F.normalize(text_features, dim=-1)

        return text_features

class ImageEncoder(nn.Module):
    """CLIP Image Encoder (Vision Transformer)"""
    def __init__(self):
        super().__init__()

        # Patch embedding
        self.patch_size = config.clip_patch_size
        self.num_patches = (config.clip_image_size // config.clip_patch_size) ** 2
        patch_dim = 3 * config.clip_patch_size * config.clip_patch_size

        self.patch_embed = nn.Linear(patch_dim, config.clip_embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.clip_embed_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, config.clip_embed_dim))

        self.transformer = nn.ModuleList([
            TransformerBlock(config.clip_embed_dim, config.clip_num_heads,
                           config.clip_mlp_ratio, config.clip_dropout)
            for _ in range(config.clip_vision_layers)
        ])

        self.norm = nn.LayerNorm(config.clip_embed_dim)

    def forward(self, images):
        B = images.shape[0]

        # Patchify
        x = rearrange(images, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                     p1=self.patch_size, p2=self.patch_size)

        # Patch embedding
        x = self.patch_embed(x)

        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Add position embedding
        x = x + self.pos_embedding

        # Transformer
        for block in self.transformer:
            x = block(x)

        x = self.norm(x)

        # Take CLS token
        image_features = x[:, 0, :]

        # Normalize
        image_features = F.normalize(image_features, dim=-1)

        return image_features

class CLIP(nn.Module):
    """Complete CLIP model"""
    def __init__(self):
        super().__init__()

        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()

        # Temperature parameter for contrastive learning
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, images, text_ids):
        image_features = self.image_encoder(images)
        text_features = self.text_encoder(text_ids)

        # Compute similarity
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()

        return logits_per_image, logits_per_text

# ═══════════════════════════════════════════════════════════════════════════════
# PART 3: U-NET (DIFFUSION MODEL) - SAFE VERSION
# ═══════════════════════════════════════════════════════════════════════════════

class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps):
        device = timesteps.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = timesteps[:, None] * embeddings[None, :]
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings

class ResBlock(nn.Module):
    """Residual block with time and context conditioning"""
    def __init__(self, in_channels: int, out_channels: int, time_emb_dim: int,
                 dropout: float = 0.1, use_checkpoint: bool = False):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

        self.time_emb = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )

        self.norm2 = nn.GroupNorm(32, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.act = nn.SiLU()

    def _forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)

        # Add time embedding
        h = h + self.time_emb(t_emb)[:, :, None, None]

        h = self.act(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)

        return h + self.skip(x)

    def forward(self, x, t_emb):
        if self.use_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward, x, t_emb)
        else:
            return self._forward(x, t_emb)

class SpatialTransformer(nn.Module):
    """Spatial transformer for cross-attention with text"""
    def __init__(self, channels: int, context_dim: int, num_heads: int = 8):
        super().__init__()
        self.norm = nn.GroupNorm(32, channels)
        self.proj_in = nn.Conv2d(channels, channels, 1)

        self.transformer_blocks = nn.ModuleList([
            CrossAttentionBlock(channels, context_dim, num_heads)
        ])

        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, context):
        B, C, H, W = x.shape
        x_in = x

        x = self.norm(x)
        x = self.proj_in(x)

        # Reshape for attention
        x = rearrange(x, 'b c h w -> b (h w) c')

        for block in self.transformer_blocks:
            x = block(x, context)

        # Reshape back
        x = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
        x = self.proj_out(x)

        return x + x_in

class CrossAttentionBlock(nn.Module):
    """Cross-attention block"""
    def __init__(self, dim: int, context_dim: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn1 = MultiHeadAttention(dim, num_heads)  # Self-attention

        self.norm2 = nn.LayerNorm(dim)
        self.attn2 = CrossAttention(dim, context_dim, num_heads)  # Cross-attention

        self.norm3 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x, context):
        x = x + self.attn1(self.norm1(x))
        x = x + self.attn2(self.norm2(x), context)
        x = x + self.mlp(self.norm3(x))
        return x

class CrossAttention(nn.Module):
    """Cross-attention mechanism"""
    def __init__(self, query_dim: int, context_dim: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = query_dim // num_heads

        self.to_q = nn.Linear(query_dim, query_dim)
        self.to_k = nn.Linear(context_dim, query_dim)
        self.to_v = nn.Linear(context_dim, query_dim)
        self.to_out = nn.Linear(query_dim, query_dim)

    def forward(self, x, context):
        B, N, C = x.shape

        q = self.to_q(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.to_k(context).reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(context).reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # 🛡️ SAFETY: FP32 for stability
        with torch.cuda.amp.autocast(enabled=False):
            q, k = q.float(), k.float()
            attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
            attn = F.softmax(attn, dim=-1)
            attn = attn.to(v.dtype)

        out = (attn @ v).permute(0, 2, 1, 3).reshape(B, N, C)
        out = self.to_out(out)

        return out

class UNet(nn.Module):
    """Diffusion U-Net with cross-attention"""
    def __init__(self):
        super().__init__()

        # Time embedding
        time_emb_dim = config.unet_model_channels * 4
        self.time_embed = nn.Sequential(
            TimestepEmbedding(config.unet_model_channels),
            nn.Linear(config.unet_model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )

        # Initial convolution
        self.conv_in = nn.Conv2d(config.unet_in_channels, config.unet_model_channels, 3, padding=1)

        # Downsampling
        self.down_blocks = nn.ModuleList([])
        channels = [config.unet_model_channels]
        now_channels = config.unet_model_channels

        for level, mult in enumerate(config.unet_channel_mult):
            out_channels = config.unet_model_channels * mult

            for _ in range(config.unet_num_res_blocks):
                self.down_blocks.append(nn.ModuleList([
                    ResBlock(now_channels, out_channels, time_emb_dim,
                            config.unet_dropout, config.unet_use_checkpoint),
                    SpatialTransformer(out_channels, config.unet_context_dim, config.unet_num_heads)
                    if config.unet_image_size // (2 ** level) in config.unet_attention_resolutions else None
                ]))
                now_channels = out_channels
                channels.append(now_channels)

            # Downsample
            if level != len(config.unet_channel_mult) - 1:
                self.down_blocks.append(nn.ModuleList([
                    nn.Conv2d(now_channels, now_channels, 3, stride=2, padding=1),
                    None
                ]))
                channels.append(now_channels)

        # Middle
        self.mid_block1 = ResBlock(now_channels, now_channels, time_emb_dim,
                                   config.unet_dropout, config.unet_use_checkpoint)
        self.mid_attn = SpatialTransformer(now_channels, config.unet_context_dim, config.unet_num_heads)
        self.mid_block2 = ResBlock(now_channels, now_channels, time_emb_dim,
                                   config.unet_dropout, config.unet_use_checkpoint)

        # Upsampling
        self.up_blocks = nn.ModuleList([])

        for level, mult in enumerate(reversed(config.unet_channel_mult)):
            out_channels = config.unet_model_channels * mult

            for i in range(config.unet_num_res_blocks + 1):
                self.up_blocks.append(nn.ModuleList([
                    ResBlock(now_channels + channels.pop(), out_channels, time_emb_dim,
                            config.unet_dropout, config.unet_use_checkpoint),
                    SpatialTransformer(out_channels, config.unet_context_dim, config.unet_num_heads)
                    if config.unet_image_size // (2 ** (len(config.unet_channel_mult) - 1 - level)) in config.unet_attention_resolutions else None
                ]))
                now_channels = out_channels

            # Upsample
            if level != len(config.unet_channel_mult) - 1:
                self.up_blocks.append(nn.ModuleList([
                    nn.ConvTranspose2d(now_channels, now_channels, 4, stride=2, padding=1),
                    None
                ]))

        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(32, now_channels),
            nn.SiLU(),
            nn.Conv2d(now_channels, config.unet_out_channels, 3, padding=1)
        )

    def forward(self, x, timesteps, context):
        # Time embedding
        t_emb = self.time_embed(timesteps)

        # Initial conv
        h = self.conv_in(x)

        # Store skip connections
        skips = [h]

        # Downsample
        for block, attn in self.down_blocks:
            if isinstance(block, nn.Conv2d):
                h = block(h)
            else:
                h = block(h, t_emb)
                if attn is not None:
                    h = attn(h, context)
            skips.append(h)

        # Middle
        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h, context)
        h = self.mid_block2(h, t_emb)

        # Upsample
        for block, attn in self.up_blocks:
            if isinstance(block, nn.ConvTranspose2d):
                h = block(h)
            else:
                h = torch.cat([h, skips.pop()], dim=1)
                h = block(h, t_emb)
                if attn is not None:
                    h = attn(h, context)

        # Output
        return self.out(h)

# ═══════════════════════════════════════════════════════════════════════════════
# MODEL INSTANTIATION & PARAMETER COUNT
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("INITIALIZING MODELS")
print("="*80)

# Create models
vae_model = VAE().to(device)
clip_model = CLIP().to(device)
unet_model = UNet().to(device)

# Count parameters
vae_params = count_parameters(vae_model)
clip_params = count_parameters(clip_model)
unet_params = count_parameters(unet_model)

print(f"\n✓ VAE initialized")
print(f"  Parameters: {vae_params:,} ({vae_params/1e6:.2f}M)")
print(f"  Architecture: 256px → 32px (8x compression)")

print(f"\n✓ CLIP initialized")
print(f"  Text Encoder: {count_parameters(clip_model.text_encoder):,}")
print(f"  Image Encoder: {count_parameters(clip_model.image_encoder):,}")
print(f"  Total Parameters: {clip_params:,} ({clip_params/1e6:.2f}M)")

print(f"\n✓ U-Net initialized")
print(f"  Parameters: {unet_params:,} ({unet_params/1e6:.2f}M)")
print(f"  Gradient Checkpointing: {config.unet_use_checkpoint}")

print(f"\n📊 Total Parameters (all models): {(vae_params + clip_params + unet_params)/1e6:.2f}M")
print("\n" + "="*80)
print("Proceed to Cell 4 (Training Utilities)")
print("="*80)

In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                    Cell 4: Training Engine & Utilities (Safe)                ║
╚═══════════════════════════════════════════════════════════════════════════════╝

Complete training engine with:
- Component-specific trainers (VAE, CLIP, U-Net)
- Safety rails (gradient clipping, FP32 loss, accumulation)
- Sampling & visualization
- Checkpointing with sliding window
- DDIM/DDPM samplers
"""

# ═══════════════════════════════════════════════════════════════════════════════
# PART 1: DIFFUSION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

class DDPMScheduler:
    """DDPM noise scheduler"""
    def __init__(self, num_steps: int = 1000, beta_start: float = 0.0001, beta_end: float = 0.02):
        self.num_steps = num_steps

        # Linear beta schedule
        self.betas = torch.linspace(beta_start, beta_end, num_steps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        # Calculations for diffusion q(x_t | x_{t-1})
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_log_variance = torch.log(torch.clamp(self.posterior_variance, min=1e-20))
        self.posterior_mean_coef1 = self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_mean_coef2 = (1.0 - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1.0 - self.alphas_cumprod)

    def add_noise(self, x_start, noise, timesteps):
        """Forward diffusion: q(x_t | x_0)"""
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod[timesteps]
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod[timesteps]

        # Reshape for broadcasting
        while len(sqrt_alpha_cumprod.shape) < len(x_start.shape):
            sqrt_alpha_cumprod = sqrt_alpha_cumprod.unsqueeze(-1)
            sqrt_one_minus_alpha_cumprod = sqrt_one_minus_alpha_cumprod.unsqueeze(-1)

        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise

    def sample_prev_timestep(self, model_output, timestep, sample):
        """Reverse diffusion: p(x_{t-1} | x_t)"""
        # Get parameters
        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod_prev[timestep]
        beta_prod_t = 1 - alpha_prod_t

        # Predict x_0
        pred_original_sample = (sample - torch.sqrt(beta_prod_t) * model_output) / torch.sqrt(alpha_prod_t)

        # Clip
        pred_original_sample = torch.clamp(pred_original_sample, -1, 1)

        # Compute mean
        pred_sample_direction = torch.sqrt(1 - alpha_prod_t_prev) * model_output
        prev_sample_mean = torch.sqrt(alpha_prod_t_prev) * pred_original_sample + pred_sample_direction

        # Add noise (except for t=0)
        variance = 0
        if timestep > 0:
            noise = torch.randn_like(model_output)
            variance = torch.sqrt(self.posterior_variance[timestep]) * noise

        prev_sample = prev_sample_mean + variance

        return prev_sample

class FlowMatchingScheduler:
    """
    Rectified Flow / Flow Matching scheduler.

    Based on "Flow Matching for Generative Modeling" (Lipman et al., 2023)
    and "Flow Straight and Fast" (Liu et al., 2023)
    """
    def __init__(self, num_train_steps: int = 1000, sigma_min: float = 0.0):
        self.num_train_steps = num_train_steps
        self.sigma_min = sigma_min  # Minimum noise level (usually 0)

    def add_noise(self, x_start, noise, timesteps):
        """
        Forward process: Linear interpolation from noise to data

        x_t = (1 - t) * noise + t * x_start

        This is simpler than DDPM's sqrt schedule!
        """
        # Normalize timesteps to [0, 1]
        t = timesteps.float() / self.num_train_steps

        # Reshape for broadcasting
        while len(t.shape) < len(x_start.shape):
            t = t.unsqueeze(-1)

        # Linear interpolation
        x_t = (1 - t) * noise + t * x_start

        return x_t

    def get_velocity(self, x_start, noise):
        """
        Compute target velocity: v = x_1 - x_0

        In flow matching, we learn the velocity field that
        transports noise to data.
        """
        return x_start - noise

    def sample_prev_timestep(self, model_output, timestep, sample, num_inference_steps=50):
        """
        Euler integration for sampling

        x_{t-dt} = x_t + v_t * dt

        This is the ODE solver for the learned flow.
        """
        # Calculate dt (time step)
        dt = 1.0 / num_inference_steps

        # Euler step: move along the predicted velocity
        prev_sample = sample + model_output * dt

        return prev_sample

    def sample_heun(self, model, model_output, timestep, sample, context, num_inference_steps=50):
        """
        Heun's method (2nd order ODE solver) - Better quality than Euler

        This is optional but gives better samples at low NFE.
        """
        dt = 1.0 / num_inference_steps

        # First Euler step
        x_temp = sample + model_output * dt

        # 2. Get velocity at the NEXT time step (Forward!)
        # We are moving UP in time, so we ADD the step size
        step_size_in_timesteps = self.num_train_steps / num_inference_steps
        t_next = timestep + step_size_in_timesteps

        # Clamp to avoid going past T_max
        t_next = torch.clamp(t_next, max=self.num_train_steps - 1)

        # Broadcast t_next
        t_next_tensor = torch.tensor([t_next] * sample.shape[0], device=sample.device)

        # Predict velocity at x_temp, t_next
        v_temp = model(x_temp, t_next_tensor, context)

        # 3. Average Velocity
        v_avg = (model_output + v_temp) / 2

        # 4. Final Step
        prev_sample = sample + v_avg * dt

        return prev_sample


class ReflowScheduler(FlowMatchingScheduler):
    """
    Reflow: Iterative refinement of flow matching

    After training a flow matching model, you can "reflow" it by:
    1. Generate samples with the trained model
    2. Use those samples as new training data
    3. Train a new model on straighter paths

    This is advanced - implement after basic flow matching works!
    """
    def __init__(self, num_train_steps: int = 1000):
        super().__init__(num_train_steps)
        self.reflow_iteration = 0  # Track which reflow iteration

    def reflow_coupling(self, model, x_0, x_1, num_steps=10):
        """
        Generate coupled (x_0, x_1) pairs for reflow training

        This straightens the learned trajectories.
        """
        # Start from x_0 (noise)
        x_t = x_0.clone()

        # Simulate the flow
        for step in range(num_steps):
            t = torch.tensor([step * self.num_train_steps // num_steps] * x_0.shape[0])
            t = t.to(x_0.device)

            # Get velocity prediction
            with torch.no_grad():
                v_pred = model(x_t, t, context=None)

            # Euler step
            dt = 1.0 / num_steps
            x_t = x_t + v_pred * dt

        # x_t is now the endpoint (new x_1)
        return x_0, x_t

class DDIMScheduler:
    """DDIM sampler - faster sampling"""
    def __init__(self, num_train_steps: int = 1000, num_inference_steps: int = 50):
        self.num_train_steps = num_train_steps
        self.num_inference_steps = num_inference_steps

        # Create timestep schedule
        self.timesteps = torch.linspace(num_train_steps - 1, 0, num_inference_steps, dtype=torch.long)

        # Beta schedule
        beta_start, beta_end = 0.0001, 0.02
        betas = torch.linspace(beta_start, beta_end, num_train_steps)
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)

    def sample_prev_timestep(self, model_output, timestep, sample, eta=0.0):
        """DDIM sampling step"""
        prev_timestep = timestep - self.num_train_steps // self.num_inference_steps

        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else torch.tensor(1.0)

        beta_prod_t = 1 - alpha_prod_t

        # Predict x_0
        pred_original_sample = (sample - torch.sqrt(beta_prod_t) * model_output) / torch.sqrt(alpha_prod_t)
        pred_original_sample = torch.clamp(pred_original_sample, -1, 1)

        # Direction pointing to x_t
        pred_sample_direction = torch.sqrt(1 - alpha_prod_t_prev) * model_output

        prev_sample = torch.sqrt(alpha_prod_t_prev) * pred_original_sample + pred_sample_direction

        return prev_sample

# ═══════════════════════════════════════════════════════════════════════════════
# PART 2: COMPONENT TRAINERS (WITH SAFETY RAILS)
# ═══════════════════════════════════════════════════════════════════════════════

class VAETrainer:
    """Trainer for VAE with safe loss computation"""
    def __init__(self, model, device, total_steps):
        self.model = model
        self.device = device
        self.logger = Logger(LOGS_DIR, "vae_training")

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        # Scheduler
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=total_steps
        )

        # AMP
        self.scaler = GradScaler() if config.use_amp else None

        # EMA
        self.ema = EMA(model, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    def compute_loss(self, recon, target, mu, logvar, kl_weight=1.0):
        """VAE loss with Free Bits (🛡️ SAFE: FP32 computation)"""

        # 🛡️ SAFETY RAIL 1: Force FP32 for Loss Computation
        with torch.cuda.amp.autocast(enabled=False):
            recon = recon.float()
            target = target.float()
            mu = mu.float()
            logvar = logvar.float()

            # Reconstruction loss (MSE)
            recon_loss = F.mse_loss(recon, target, reduction='mean')

            # KL divergence per sample, per latent dimension
            kl_per_latent = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())

            # Free bits: Only penalize KL above a threshold
            free_bits_threshold = self.model.free_bits / np.prod(kl_per_latent.shape[1:])
            kl_per_latent = torch.clamp(kl_per_latent, min=free_bits_threshold)

            # Average over batch
            kl_loss = kl_per_latent.mean()

            # Total loss with annealed KL weight
            total_loss = recon_loss + kl_weight * kl_loss

        return total_loss, recon_loss, kl_loss

    def get_kl_weight(self, step, mode='cyclical'):
        """KL weight annealing schedule"""
        if mode == 'constant':
            return config.vae_beta

        elif mode == 'linear':
            warmup_steps = 10000
            return min(1.0, step / warmup_steps) * config.vae_beta

        elif mode == 'cyclical':
            cycle_length = 5000
            cycle_progress = (step % cycle_length) / cycle_length

            if cycle_progress < 0.5:
                kl_weight = (cycle_progress * 2) * config.vae_beta
            else:
                kl_weight = config.vae_beta

            return kl_weight

        return config.vae_beta

    def train_step(self, batch):
        """Single training step with gradient accumulation"""
        self.model.train()
        images = batch["image"].to(self.device)

        # Get annealed KL weight
        kl_weight = self.get_kl_weight(self.global_step, mode='cyclical')

        # Forward pass
        if config.use_amp:
            with autocast():
                recon, mu, logvar = self.model(images)
        else:
            recon, mu, logvar = self.model(images)

        # 🛡️ SAFETY: Compute loss in FP32
        loss, recon_loss, kl_loss = self.compute_loss(
            recon, images, mu, logvar, kl_weight=kl_weight
        )

        # 🛡️ SAFETY RAIL 2: Gradient Accumulation
        loss = loss / config.accumulation_steps

        # Backward pass
        if config.use_amp:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        # 🛡️ SAFETY RAIL 3: Optimizer Step with Gradient Clipping
        if (self.global_step + 1) % config.accumulation_steps == 0:
            if config.use_amp:
                self.scaler.unscale_(self.optimizer)

            # Gradient clipping to prevent explosions
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), config.gradient_clip)

            if config.use_amp:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()

            self.optimizer.zero_grad()
            self.scheduler.step()

            if self.ema:
                self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item() * config.accumulation_steps,
            "recon_loss": recon_loss.item(),
            "kl_loss": kl_loss.item(),
            "kl_weight": kl_weight,
            "lr": self.optimizer.param_groups[0]['lr']
        }

    @torch.no_grad()
    def sample(self, num_samples=4):
        """Generate reconstruction samples"""
        self.model.eval()

        # Get validation batch
        val_loader = get_dataloader("vae", batch_size=num_samples)[1]
        batch = next(iter(val_loader))
        images = batch["image"].to(self.device)

        # Apply EMA if available
        if self.ema:
            self.ema.apply_shadow()

        # Reconstruct
        recon, _, _ = self.model(images)

        # Restore weights
        if self.ema:
            self.ema.restore()

        # Denormalize
        images = (images + 1) / 2
        recon = (recon + 1) / 2

        # Create grid
        comparison = torch.cat([images, recon], dim=0)
        grid = make_grid(comparison, nrow=num_samples)

        return grid

class CLIPTrainer:
    """Trainer for CLIP with safe attention"""
    def __init__(self, model, tokenizer, device, total_steps):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.logger = Logger(LOGS_DIR, "clip_training")

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=total_steps
        )

        self.scaler = GradScaler() if config.use_amp else None
        self.ema = EMA(model, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    def compute_loss(self, logits_per_image, logits_per_text):
        """Contrastive loss (🛡️ SAFE: FP32 computation)"""
        with torch.cuda.amp.autocast(enabled=False):
            logits_per_image = logits_per_image.float()
            logits_per_text = logits_per_text.float()

            batch_size = logits_per_image.shape[0]
            labels = torch.arange(batch_size, device=self.device)

            loss_i = F.cross_entropy(logits_per_image, labels)
            loss_t = F.cross_entropy(logits_per_text, labels)

            loss = (loss_i + loss_t) / 2

            # Accuracy
            acc_i = (logits_per_image.argmax(dim=1) == labels).float().mean()
            acc_t = (logits_per_text.argmax(dim=1) == labels).float().mean()

        return loss, acc_i, acc_t

    def train_step(self, batch):
        """Single training step with gradient accumulation"""
        self.model.train()

        images = batch["image"].to(self.device)
        captions = batch["caption"]

        # Tokenize captions
        text_ids = torch.stack([self.tokenizer.encode(cap) for cap in captions]).to(self.device)

        # Forward pass
        if config.use_amp:
            with autocast():
                logits_per_image, logits_per_text = self.model(images, text_ids)
        else:
            logits_per_image, logits_per_text = self.model(images, text_ids)

        # Compute loss in FP32
        loss, acc_i, acc_t = self.compute_loss(logits_per_image, logits_per_text)

        # Gradient accumulation
        loss = loss / config.accumulation_steps

        # Backward pass
        if config.use_amp:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step with gradient clipping
        if (self.global_step + 1) % config.accumulation_steps == 0:
            if config.use_amp:
                self.scaler.unscale_(self.optimizer)

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), config.gradient_clip)

            if config.use_amp:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()

            self.optimizer.zero_grad()
            self.scheduler.step()

            if self.ema:
                self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item() * config.accumulation_steps,
            "acc_image": acc_i.item(),
            "acc_text": acc_t.item(),
            "lr": self.optimizer.param_groups[0]['lr']
        }

class UNetTrainer:
    """Trainer for U-Net with Flow Matching"""
    def __init__(self, unet, vae, clip, tokenizer, device, total_steps, use_flow_matching=True):
        self.unet = unet
        self.model = unet
        self.vae = vae
        self.clip = clip
        self.tokenizer = tokenizer
        self.device = device
        self.logger = Logger(LOGS_DIR, "unet_training")
        self.use_flow_matching = use_flow_matching

        # Freeze VAE and CLIP
        self.vae.eval()
        self.clip.eval()
        for param in self.vae.parameters():
            param.requires_grad = False
        for param in self.clip.parameters():
            param.requires_grad = False

        # Choose scheduler based on mode
        if use_flow_matching:
            self.noise_scheduler = FlowMatchingScheduler(config.num_diffusion_steps)
            self.logger.log("✅ Using Flow Matching scheduler")
        else:
            self.noise_scheduler = DDPMScheduler(config.num_diffusion_steps)
            self.logger.log("✅ Using DDPM scheduler")

        # Move scheduler tensors to device (for DDPM)
        if not use_flow_matching:
            scheduler_tensors = [
                'betas', 'alphas', 'alphas_cumprod', 'alphas_cumprod_prev',
                'sqrt_alphas_cumprod', 'sqrt_one_minus_alphas_cumprod',
                'posterior_variance', 'posterior_log_variance',
                'posterior_mean_coef1', 'posterior_mean_coef2'
            ]
            for tensor_name in scheduler_tensors:
                if hasattr(self.noise_scheduler, tensor_name):
                    tensor = getattr(self.noise_scheduler, tensor_name)
                    setattr(self.noise_scheduler, tensor_name, tensor.to(device))

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                unet.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                unet.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=total_steps
        )

        self.scaler = GradScaler() if config.use_amp else None
        self.ema = EMA(unet, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    @torch.no_grad()
    def encode_images(self, images):
        """Encode images to latent space"""
        z, _, _ = self.vae.encode(images)
        return z

    @torch.no_grad()
    def encode_text(self, captions):
        """Encode text to embeddings"""
        text_ids = torch.stack([self.tokenizer.encode(cap) for cap in captions]).to(self.device)
        text_features = self.clip.text_encoder(text_ids)
        text_features = text_features.unsqueeze(1)
        return text_features

    def train_step(self, batch):
        """
        Single training step with Flow Matching or DDPM

        Key difference:
        - DDPM: Learn to predict noise ε
        - Flow Matching: Learn to predict velocity v = x_1 - x_0
        """
        self.unet.train()

        images = batch["image"].to(self.device)
        captions = batch["caption"]
        batch_size = images.shape[0]

        # Encode images to latent space
        with torch.no_grad():
            latents = self.encode_images(images)  # x_1 (data)

            # Classifier-free guidance: randomly drop text conditioning
            if random.random() < config.cfg_dropout:
                text_embeddings = torch.zeros(batch_size, 1, config.clip_embed_dim).to(self.device)
            else:
                text_embeddings = self.encode_text(captions)

        # Sample noise
        noise = torch.randn_like(latents)  # x_0 (noise)

        # Sample timesteps
        timesteps = torch.randint(
            0, config.num_diffusion_steps,
            (batch_size,), device=self.device
        ).long()

        # Add noise to latents
        noisy_latents = self.noise_scheduler.add_noise(latents, noise, timesteps)

        # Forward pass
        if config.use_amp:
            with autocast():
                model_output = self.unet(noisy_latents, timesteps, text_embeddings)
        else:
            model_output = self.unet(noisy_latents, timesteps, text_embeddings)

        # 🔥 KEY DIFFERENCE: Compute loss based on scheduler type
        with torch.cuda.amp.autocast(enabled=False):
            if self.use_flow_matching:
                # Flow Matching: Predict velocity v = x_1 - x_0
                target_velocity = self.noise_scheduler.get_velocity(latents, noise)
                loss = F.mse_loss(model_output.float(), target_velocity.float())
            else:
                # DDPM: Predict noise ε
                loss = F.mse_loss(model_output.float(), noise.float())

        # Gradient accumulation
        loss = loss / config.accumulation_steps

        # Backward pass
        if config.use_amp:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step with gradient clipping
        if (self.global_step + 1) % config.accumulation_steps == 0:
            if config.use_amp:
                self.scaler.unscale_(self.optimizer)

            torch.nn.utils.clip_grad_norm_(self.unet.parameters(), config.gradient_clip)

            if config.use_amp:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()

            self.optimizer.zero_grad()
            self.scheduler.step()

            if self.ema:
                self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item() * config.accumulation_steps,
            "lr": self.optimizer.param_groups[0]['lr']
        }

    @torch.no_grad()
    def sample(self, prompts, num_inference_steps=20, guidance_scale=7.5, use_heun=False):
        """
        Generate images from text prompts

        Flow Matching allows much fewer steps (10-20 vs 50-1000 for DDPM)!
        """
        self.unet.eval()

        if self.ema:
            self.ema.apply_shadow()

        batch_size = len(prompts)

        # Encode text
        text_embeddings = self.encode_text(prompts)

        # For classifier-free guidance
        uncond_embeddings = torch.zeros_like(text_embeddings)
        text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

        # Initialize latents (pure noise, x_0)
        latents = torch.randn(
            batch_size, config.unet_in_channels,
            config.unet_image_size, config.unet_image_size
        ).to(self.device)

        # 🔥 FLOW MATCHING SAMPLING
        if self.use_flow_matching:
            # Create timestep schedule (from T to 0)
            timesteps = torch.linspace(
                0, config.num_diffusion_steps - 1,
                num_inference_steps
            ).long().to(self.device)

            for i, t in enumerate(tqdm(timesteps, desc="Sampling (Flow)")):
                # Expand for CFG
                latent_model_input = torch.cat([latents] * 2)
                t_input = torch.tensor([t] * (batch_size * 2), device=self.device)

                # Predict velocity
                velocity_pred = self.unet(latent_model_input, t_input, text_embeddings)

                # Classifier-free guidance
                velocity_uncond, velocity_text = velocity_pred.chunk(2)
                velocity_pred = velocity_uncond + guidance_scale * (velocity_text - velocity_uncond)

                # Integrate (Euler or Heun method)
                if use_heun and i < len(timesteps) - 1:
                    # Pass embeddings as context for the second model call in Heun
                    # Note: We pass the FULL embeddings (including uncond) inside Heun helper usually,
                    # but for simplicity in your helper, you might need to handle CFG inside Heun
                    # OR just use Euler for now if Heun is too complex to wire up with CFG.
                    #
                    # SIMPLIFIED FIX for now: Just use Euler if unsure, but if using Heun:
                    # You need to pass a way to calculate CFG inside sample_heun.
                    # For now, let's stick to Euler or assume sample_heun takes a CFG-capable callable.

                    # Let's revert to Euler for safety unless you update sample_heun to handle CFG
                    latents = self.noise_scheduler.sample_prev_timestep(
                        velocity_pred, t, latents, num_inference_steps
                    )
                else:
                    # Euler method
                    latents = self.noise_scheduler.sample_prev_timestep(
                        velocity_pred, t, latents, num_inference_steps
                    )

        # 🔥 DDPM SAMPLING (Your original code)
        else:
            scheduler = DDIMScheduler(config.num_diffusion_steps, num_inference_steps)
            timesteps = scheduler.timesteps.to(self.device)

            for t in tqdm(timesteps, desc="Sampling (DDPM)"):
                latent_model_input = torch.cat([latents] * 2)
                t_input = torch.tensor([t] * (batch_size * 2), device=self.device)

                noise_pred = self.unet(latent_model_input, t_input, text_embeddings)

                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

                latents = scheduler.sample_prev_timestep(noise_pred, t, latents)

        # Decode latents to images
        images = self.vae.decode(latents)
        images = (images + 1) / 2
        images = torch.clamp(images, 0, 1)

        if self.ema:
            self.ema.restore()

        return images

# ═══════════════════════════════════════════════════════════════════════════════
# PART 3: MAIN TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

def train_component(component: str):
    """Main training function with sliding window checkpoints and GPU augmentations"""

    print("\n" + "="*80)
    print(f"TRAINING {component.upper()}")
    print("="*80)

    # Get dataloader
    train_loader, val_loader = get_dataloader(component, config.batch_size)

    # 🚀 Initialize GPU Augmentation Pipeline
    print(f"\n⚡ Initializing GPU augmentation pipeline...")
    gpu_aug = GPUAugmentations(component).to(device)
    gpu_aug.eval()  # Set to eval mode (no dropout in augmentations)
    print(f"✓ GPU augmentations ready (Kornia pipeline active)")

    total_steps = (len(train_loader) * config.num_epochs) // config.accumulation_steps
    print(f"✓ Total training steps: {total_steps:,}")

    # Create trainer
    if component == "vae":
        trainer = VAETrainer(vae_model, device, total_steps)

    elif component == "clip":
        trainer = CLIPTrainer(clip_model, tokenizer, device, total_steps)

    elif component == "unet":
        # Load pretrained VAE and CLIP
        vae_checkpoint = torch.load(config.vae_path, map_location=device)
        clip_checkpoint = torch.load(config.clip_path, map_location=device)

        if "model_state_dict" in vae_checkpoint:
            vae_model.load_state_dict(vae_checkpoint["model_state_dict"])
        else:
            vae_model.load_state_dict(vae_checkpoint)

        if "model_state_dict" in clip_checkpoint:
            clip_model.load_state_dict(clip_checkpoint["model_state_dict"])
        else:
            clip_model.load_state_dict(clip_checkpoint)

        print(f"✓ Loaded VAE from {config.vae_path}")
        print(f"✓ Loaded CLIP from {config.clip_path}")

        trainer = UNetTrainer(unet_model, vae_model, clip_model, tokenizer, device, total_steps, use_flow_matching=config.use_flow_matching)

    else:
        raise ValueError(f"Unknown component: {component}")

    # Load checkpoint if resuming
    if config.checkpoint_path and os.path.exists(config.checkpoint_path):
        trainer.epoch, trainer.global_step, _ = load_checkpoint(
            config.checkpoint_path,
            trainer.model,
            trainer.optimizer,
            trainer.scheduler,
            trainer.ema if trainer.ema else None,
            trainer.scaler
        )

    # Training loop
    print(f"\nStarting training...")
    print(f"  Epochs: {config.num_epochs}")
    print(f"  Batch size: {config.batch_size} (effective: {config.batch_size * config.accumulation_steps})")
    print(f"  Total steps: {len(train_loader) * config.num_epochs}")
    print(f"  Device: {device}")

    start_time = time.time()
    best_loss = float('inf')

    for epoch in range(trainer.epoch, config.num_epochs):
        trainer.epoch = epoch
        epoch_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

        for batch in pbar:
            # 🚀 STEP 1: Move raw data to GPU (Fast PCIe transfer)
            images = batch["image"].to(device, non_blocking=True)

            # 🚀 STEP 2: Apply GPU augmentations (Kornia - parallel processing)
            # This replaces ~25M CPU ops with GPU-accelerated operations
            with torch.no_grad():
                images = gpu_aug(images)

            # Update batch with augmented images
            batch["image"] = images

            # 🚀 STEP 3: Train (existing pipeline)
            metrics = trainer.train_step(batch)
            epoch_loss += metrics["loss"]

            # Update progress bar
            pbar.set_postfix({k: f"{v:.6f}" for k, v in metrics.items()})

            # Logging
            if trainer.global_step % config.log_every == 0:
                trainer.logger.log_metrics(trainer.global_step, metrics)

            # Sampling
            if trainer.global_step % config.sample_every == 0:
                trainer.logger.log(f"Generating samples at step {trainer.global_step}...")

                if component in ["vae"]:
                    sample_grid = trainer.sample(config.num_samples)
                    save_image(
                        sample_grid,
                        os.path.join(SAMPLES_DIR, f"{component}_step_{trainer.global_step:06d}.png")
                    )

                elif component == "unet":
                    test_prompts = [
                        "a dog on a skateboard",
                        "a cat sitting on a table",
                        "a beautiful sunset over mountains",
                        "a person playing guitar"
                    ]
                    sample_images = trainer.sample(test_prompts[:config.num_samples])
                    save_image(
                        sample_images,
                        os.path.join(SAMPLES_DIR, f"{component}_step_{trainer.global_step:06d}.png"),
                        nrow=2
                    )

            # Checkpointing with sliding window
            if trainer.global_step % config.save_every == 0:
                checkpoint_path = os.path.join(
                    CHECKPOINTS_DIR,
                    f"{component}_step_{trainer.global_step:06d}.pt"
                )
                save_checkpoint(
                    trainer.model,
                    trainer.optimizer,
                    trainer.scheduler,
                    epoch,
                    trainer.global_step,
                    metrics["loss"],
                    checkpoint_path,
                    component,  # Pass component for sliding window
                    trainer.ema if trainer.ema else None,
                    trainer.scaler
                )

                # Save best model (not subject to sliding window)
                if metrics["loss"] < best_loss:
                    best_loss = metrics["loss"]
                    best_path = os.path.join(CHECKPOINTS_DIR, f"{component}_best.pt")
                    save_checkpoint(
                        trainer.model,
                        trainer.optimizer,
                        trainer.scheduler,
                        epoch,
                        trainer.global_step,
                        metrics["loss"],
                        best_path,
                        component,
                        trainer.ema if trainer.ema else None,
                        trainer.scaler
                    )

        # Epoch summary
        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start_time

        trainer.logger.log(
            f"Epoch {epoch+1} complete - Avg Loss: {avg_loss:.6f} - Time: {format_time(elapsed)}"
        )

        # Plot metrics
        if component == "vae":
            trainer.logger.plot_metrics(["loss", "recon_loss", "kl_loss"])
        elif component == "clip":
            trainer.logger.plot_metrics(["loss", "acc_image", "acc_text"])
        else:
            trainer.logger.plot_metrics(["loss"])

    # Final save
    final_path = os.path.join(CHECKPOINTS_DIR, f"{component}_final.pt")
    save_checkpoint(
        trainer.model,
        trainer.optimizer,
        trainer.scheduler,
        config.num_epochs,
        trainer.global_step,
        avg_loss,
        final_path,
        component,
        trainer.ema if trainer.ema else None,
        trainer.scaler
    )

    # Save EMA model separately
    if trainer.ema:
        ema_path = os.path.join(CHECKPOINTS_DIR, f"{component}_ema.pt")
        torch.save(trainer.ema.shadow, ema_path)
        trainer.logger.log(f"✓ EMA weights saved: {ema_path}")

    total_time = time.time() - start_time
    trainer.logger.log(f"\n{'='*80}")
    trainer.logger.log(f"TRAINING COMPLETE!")
    trainer.logger.log(f"Total time: {format_time(total_time)}")
    trainer.logger.log(f"Final loss: {avg_loss:.6f}")
    trainer.logger.log(f"Best loss: {best_loss:.6f}")
    trainer.logger.log(f"{'='*80}\n")

print("\n" + "="*80)
print("TRAINING ENGINE READY")
print("="*80)
print("\nProceed to Cell 5 to start training!")
print("\nAvailable components:")
print("  1. 'vae' - Train VAE (256x256 → 32x32)")
print("  2. 'clip' - Train CLIP (Text-Image alignment)")
print("  3. 'unet' - Train U-Net (Diffusion model)")
print("\nExample: config.component = 'vae'")
print("         train_component(config.component)")

In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                  Cell 5: Unified Training Launcher                           ║
╚═══════════════════════════════════════════════════════════════════════════════╝

Configure and launch training for any component.
Auto-resume from latest checkpoint with sliding window management.
"""

import glob

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION - SET YOUR TARGET HERE
# ═══════════════════════════════════════════════════════════════════════════════

# Override any config if needed
# config.component = "unet"  # Options: "vae", "clip", "unet"
# config.batch_size = 16  # Uncomment to change
# config.learning_rate = 1e-5  # Uncomment to change
# config.num_epochs = 100  # Uncomment to change
# Force the scheduler to skip the 4-hour warmup
config.warmup_steps = 0

print("="*80)
print(f"🎯 TRAINING CONFIGURATION")
print("="*80)
print(f"Component:       {config.component.upper()}")
print(f"Batch Size:      {config.batch_size} (effective: {config.batch_size * config.accumulation_steps})")
print(f"Learning Rate:   {config.learning_rate}")
print(f"Epochs:          {config.num_epochs}")
print(f"Gradient Clip:   {config.gradient_clip}")
print(f"Use AMP:         {config.use_amp}")
print(f"Max Checkpoints: {config.max_checkpoints} (sliding window)")
print(f"Save Every:      {config.save_every}")
print(f"Sample Every:    {config.sample_every}")
print("="*80)

# ═══════════════════════════════════════════════════════════════════════════════
# AUTO-RESUME LOGIC
# ═══════════════════════════════════════════════════════════════════════════════

def find_latest_checkpoint(component):
    """Automatically finds the checkpoint with the highest step number"""
    pattern = os.path.join(CHECKPOINTS_DIR, f"{component}_step_*.pt")
    checkpoints = glob.glob(pattern)

    if not checkpoints:
        return None

    # Sort by step number
    try:
        latest_ckpt = max(checkpoints, key=lambda p: int(p.split("_step_")[-1].split(".")[0]))
        return latest_ckpt
    except ValueError:
        return None

# Check for existing checkpoints
latest_ckpt = find_latest_checkpoint(config.component)

if latest_ckpt:
    print(f"\n🔄 Auto-resume: Found {os.path.basename(latest_ckpt)}")
    config.checkpoint_path = latest_ckpt

    # Load step number for display
    try:
        step = int(latest_ckpt.split("_step_")[-1].split(".")[0])
        print(f"   Resuming from step {step:,}")
    except:
        pass
else:
    print(f"\n🆕 No checkpoints found. Starting from scratch (Step 0).")
    config.checkpoint_path = None

# Set paths for U-Net (requires pretrained VAE and CLIP)
if config.component == "unet":
    config.vae_path = os.path.join(CHECKPOINTS_DIR, "vae_final.pt")
    config.clip_path = os.path.join(CHECKPOINTS_DIR, "clip_final.pt")

    if not os.path.exists(config.vae_path):
        raise FileNotFoundError(f"❌ VAE checkpoint not found: {config.vae_path}")
    if not os.path.exists(config.clip_path):
        raise FileNotFoundError(f"❌ CLIP checkpoint not found: {config.clip_path}")

    print(f"\n✓ VAE checkpoint: {config.vae_path}")
    print(f"✓ CLIP checkpoint: {config.clip_path}")

# ═══════════════════════════════════════════════════════════════════════════════
# LAUNCH TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print(f"🚀 STARTING TRAINING: {config.component.upper()}")
print("="*80)

try:
    train_component(config.component)
    print(f"\n🎉 {config.component.upper()} training finished successfully!")
except KeyboardInterrupt:
    print(f"\n⚠️ Training interrupted by user.")
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    import traceback
    traceback.print_exc()